## 1. Objective

In this notebook, we study the behaviour of implied volatility in different synthetic market regimes, and provide insight on its characteristics. We can analyze:

- shape of implied volatility surface
- effects of smoothing
- error estimation
- smile characteristics
- skew characteristics
- term structure

## 2. Assumptions

- European call options
- No dividends
- Constant risk-free rate
- Synthetic volatility surface σ(K, T) is used as ground truth (Can use real market data also)

## 3. Experimental Setup

This section defines the strikes, maturities and market regime parameters.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

### Imports

In [2]:
import numpy as np
import pandas as pd

from data_generation.synthetic_data import generate_market_prices
from computation.implied_volatility import calculate_implied_volatility
from computation.smoothing import surface_smoother
from computation.error_calculation import error_calculator

from visualization.surface import plot_volatility_surface
from visualization.smile import plot_smile
from visualization.term_structure import plot_term_structure
from visualization.error_plot import plot_error_surface

### Market Grid Definition

We define a grid of strikes and maturities over which the volatility surface
will be constructed.

- Strikes span deep OTM to deep ITM
- Maturities are dense at the short end to capture event-driven behavior

In [3]:
S0 = 100.0
r = 0.04

strikes = np.linspace(70, 130, 41)

maturities = np.array([
    0.02, 0.05, 0.10, 0.15, 0.25,
    0.40, 0.50, 0.75,
    1.00, 1.50, 2.00
])

### Market Regimes

Each regime is defined by a set of parameters controlling:
- ATM volatility level
- short-term volatility bump
- skew strength
- smile curvature
- decay with maturity

These parameters are held constant across experiments unless explicitly changed.

In [4]:
baseline_params = dict(
    ATM_VOL=0.18,
    ATM_VOL_BUMP=0.08,
    ATM_DECAY=0.5,
    SKEW_STRENGTH=-0.30,
    SKEW_DECAY=0.6,
    SMILE_STRENGTH=0.60,
    SMILE_DECAY=1.2,
)

crash_fear_params = dict(
    ATM_VOL=0.20,
    ATM_VOL_BUMP=0.15,
    ATM_DECAY=0.4,
    SKEW_STRENGTH=-0.55,
    SKEW_DECAY=1.2,
    SMILE_STRENGTH=0.80,
    SMILE_DECAY=1.0,
)

calm_market_params = dict(
    ATM_VOL=0.12,
    ATM_VOL_BUMP=0.02,
    ATM_DECAY=0.4,
    SKEW_STRENGTH=-0.15,
    SKEW_DECAY=0.5,
    SMILE_STRENGTH=0.25,
    SMILE_DECAY=0.8,
)

event_shock_params = dict(
    ATM_VOL=0.16,
    ATM_VOL_BUMP=0.25,
    ATM_DECAY=0.15,
    SKEW_STRENGTH=-0.35,
    SKEW_DECAY=0.4,
    SMILE_STRENGTH=0.50,
    SMILE_DECAY=0.5,
)


### Implied Volatility Recovery (Helper Function)

Given market prices generated from the true volatility surface, we recover
implied volatility via numerical inversion of the Black–Scholes formula.

In [5]:
def recover_implied_surface(market_prices, spot, rate):
    implied = pd.DataFrame(
        index=market_prices.index,
        columns=market_prices.columns,
        dtype=float
    )

    for K in market_prices.index:
        for T in market_prices.columns:
            implied.loc[K, T] = calculate_implied_volatility(
                market_prices.loc[K, T],
                spot,
                K,
                T,
                rate
            )
    return implied

## 4. Baseline Market Regime
 This regime represents a stable equity market with moderate volatility, weak skew and smile and smooth term structure. Volatility varies gradually and consistently across strikes and maturities.

In [6]:
market_prices, true_vol = generate_market_prices(
    spot=S0,
    strikes=strikes,
    maturities=maturities,
    rate=r,
    noise_amt=0.0,
    **baseline_params
)

implied_raw = recover_implied_surface(market_prices, S0, r)
implied_smooth = surface_smoother(implied_raw)
error_surface = error_calculator(true_vol, implied_raw)
plot_volatility_surface(true_vol * 100, title="True Volatility Surface")

### Implied Volatility Surface

In [7]:
plot_volatility_surface(implied_raw * 100, title="Implied Vol Surface (Raw)")

### Smooth Implied Volatility Surface

In [8]:
plot_volatility_surface(implied_smooth * 100, title="Implied Vol Surface (Smoothed)")

### Error Surface

In [9]:
plot_error_surface(error_surface)

### Smile and Term structure

In [10]:
plot_smile(implied_smooth * 100)
plot_term_structure(implied_smooth * 100, S0)

### Observations

- Short-dated implied volatility is increased due to the At-The-Money bump.
- Negative skew raises implied volatility for low strikes.
- Smile curvature decays exponentially with maturity.
- Smoothing stabilizes local numerical noise without distorting structure.

## 5. Crash fear Market Regime

This regime represents high downside risk fear and strong demand for downside protection. It has high short-term volatility and strong negative skew.

In [11]:
market_prices, true_vol = generate_market_prices(
    spot=S0,
    strikes=strikes,
    maturities=maturities,
    rate=r,
    noise_amt=0.0,
    **crash_fear_params
)

implied_raw = recover_implied_surface(market_prices, S0, r)
implied_smooth = surface_smoother(implied_raw)
error_surface = error_calculator(true_vol, implied_raw)

plot_volatility_surface(true_vol * 100, title="True Volatility Surface")

### Implied Volatility Surface

In [12]:
plot_volatility_surface(implied_raw * 100, title="Implied Vol Surface (Raw)")


### Smooth Implied Volatility Surface

In [13]:
plot_volatility_surface(implied_smooth * 100, title="Implied Vol Surface (Smoothed)")

### Error Surface

In [14]:
plot_error_surface(error_surface)

### Smile and Term structure

In [15]:
plot_smile(implied_smooth * 100)
plot_term_structure(implied_smooth * 100, S0)

### Observations

- Short-dated implied volatility is significantly elevated, reflecting fear of downside risk.
- Strong negative skew causes implied volatility to rise sharply for lower strikes.
- Smile curvature is steep and asymmetric, more dominant in the left wing.
- The implied surface remains stable and smooth due to high option prices and strong vega.

## 6. Calm Market Regime

This regime represents a minimum volatility environment with low premiums and minimal curvature in the volatility surface. Option prices are low and sensitivity to volatility is low.

In [16]:
market_prices, true_vol = generate_market_prices(
    spot=S0,
    strikes=strikes,
    maturities=maturities,
    rate=r,
    noise_amt=0.0,
    **calm_market_params
)

implied_raw = recover_implied_surface(market_prices, S0, r)
implied_smooth = surface_smoother(implied_raw)
error_surface = error_calculator(true_vol, implied_raw)

plot_volatility_surface(true_vol * 100, title="True Volatility Surface")

### Implied Volatility Surface

In [17]:
plot_volatility_surface(implied_raw * 100, title="Implied Vol Surface (Raw)")

### Smooth Implied Volatility Surface

In [18]:
plot_volatility_surface(implied_smooth * 100, title="Implied Vol Surface (Smoothed)")

### Error Surface

In [19]:
plot_error_surface(error_surface)

### Smile and Term structure

In [20]:
plot_smile(implied_smooth * 100)
plot_term_structure(implied_smooth * 100, S0)

### Observations

- Overall implied volatility levels are low and relatively flat across maturities.
- Deep OTM and short-dated implied volatilities appear noisy or irregular with very low option prices.
- Near-zero option prices lead to numerical instability when performing implied volatility inversion.
- Smoothing improves visual stability but masks poor implied volatility regions of the surface.

## 7. Event shock Market Regime

This regime represents a market expecting an event in the short term such as an earnings report or other economic announcements. Volatility is high in short-term options and decays rapidly with time.

In [21]:
market_prices, true_vol = generate_market_prices(
    spot=S0,
    strikes=strikes,
    maturities=maturities,
    rate=r,
    noise_amt=0.0,
    **event_shock_params
)

implied_raw = recover_implied_surface(market_prices, S0, r)
implied_smooth = surface_smoother(implied_raw)
error_surface = error_calculator(true_vol, implied_raw)

plot_volatility_surface(true_vol * 100, title="True Volatility Surface")

### Implied Volatility Surface

In [22]:
plot_volatility_surface(implied_raw * 100, title="Implied Vol Surface (Raw)")

### Smooth Implied Volatility Surface

In [23]:
plot_volatility_surface(implied_smooth * 100, title="Implied Vol Surface (Smoothed)")

### Error Surface

In [24]:
plot_error_surface(error_surface)

### Smile and Term structure

In [25]:
plot_smile(implied_smooth * 100)
plot_term_structure(implied_smooth * 100, S0)

### Observations

- Implied volatility spikes sharply at short maturities and decays rapidly with time as fear of event subsides.
- Smile curvature is strongest near the event and weakens for longer maturities.
- Skew is present but dominated by term-structure effect.
- The surface clearly separates short-term event risk from long-term volatility expectations.

## 8. Common Observations across regimes
- Short-dated and low-strike price options consistently have high volatility
- Implied volatility calculation is more accurate and consistent when prices are informative
- Calmer markets have more numerical instability


## 9. Limitations

- Synthetic volatility is simple and parametric.
- No arbitrage constraints imposed.
- Noise model is simple.
- Simplified market view.